# COMP90051 — FMA-Medium Nested CV Experiment

**Before running:** Runtime → Change runtime type → **T4 GPU**

## What this notebook does
Runs nested cross-validation (outer K=10, inner K=3) for **three models × two feature sets**:
- **Feature set A** — original 518 acoustic features
- **Feature set B** — 518 original + 121 constructed = 639 features

The A vs B comparison on minority-group recall is the core experiment that answers the research question.

## Files to upload
| File | Type | Used for |
|------|------|----------|
| `kfold.py` | code | stratified K-Fold engine |
| `gnb.py` | code | Gaussian Naive Bayes |
| `lr_sgd.py` | code | Logistic Regression (SGD) |
| `resnet_tab.py` | code | FT-Transformer |
| `nested_cv.py` | code | nested CV engine (per-fold Winsorize) |
| `metrics.py` | code | accuracy / macro-F1 / minority recall |
| `run_all.py` | code | experiment runner |
| `X_medium_raw_derived.npy` | data | raw 639-dim features (pre-Winsorize) |
| `y_medium.npy` | data | genre labels (int32) |

> **Important:** Upload `X_medium_raw_derived.npy`, NOT `X_medium.npy`.
> Per-fold Winsorization is applied inside `nested_cv.py` to prevent data leakage.

## Estimated runtime (T4 GPU)
| Step | Models | Time |
|------|--------|------|
| Cell 4 | Quick sanity check | ~2 min |
| Cell 5 | GNB  A + B | ~10 min (CPU) |
| Cell 6 | LR   A + B | ~20 min (CPU) |
| Cell 7 | FT   A + B | ~120 min (GPU) |

## Cell 1 — Upload code files (7 x .py)

In [ ]:
from google.colab import files
import os

print('Select all 7 .py files at once:')
print('kfold.py, gnb.py, lr_sgd.py, resnet_tab.py, nested_cv.py, metrics.py, run_all.py')
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

## Cell 2 — Upload data files
> Upload `X_medium_raw_derived.npy` (NOT `X_medium.npy`) and `y_medium.npy`.

In [ ]:
import os
os.makedirs('data', exist_ok=True)
os.makedirs('results', exist_ok=True)

print('Select 2 files: X_medium_raw_derived.npy  and  y_medium.npy')
uploaded_data = files.upload()

for fname in uploaded_data:
    dest = f'data/{fname}'
    os.rename(fname, dest)
    print(f'  Moved  {fname}  ->  {dest}')

## Cell 3 — Verify files & GPU

In [ ]:
import numpy as np
import torch
import os

# GPU
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU model  :', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. FT-Transformer will be very slow.')

# Data
X = np.load('data/X_medium_raw_derived.npy')
y = np.load('data/y_medium.npy')
print(f'\nX_medium_raw_derived : {X.shape}  (expected: (17000, 639))')
print(f'y                    : {y.shape}  classes: {len(np.unique(y))}  (expected: 16)')

# Quick sanity: raw derived file should have extreme values (not yet Winsorized)
z_extreme = (np.abs((X - X.mean(0)) / (X.std(0) + 1e-8)) > 10).sum()
print(f'|z| > 10 count: {z_extreme}  (>0 means raw/un-Winsorized -- correct)')

# Code files
required = ['kfold.py','gnb.py','lr_sgd.py','resnet_tab.py',
            'nested_cv.py','metrics.py','run_all.py']
print()
all_ok = True
for f in required:
    status = 'OK' if os.path.exists(f) else 'MISSING'
    if status == 'MISSING':
        all_ok = False
    print(f'  {f}: {status}')

if all_ok:
    print('\nAll files present. Ready to run.')
else:
    print('\nSome files are missing — re-run Cell 1.')

## Cell 4 — Quick sanity check (~2 min)
Runs GNB on 1000 samples, outer=3, both feature sets A & B.
Confirms the full pipeline (per-fold Winsorize → fit → evaluate) has no errors.

In [ ]:
import os
os.environ['FMA_DATA_DIR']   = '/content/data'
os.environ['FMA_RESULT_DIR'] = '/content/results'

!python run_all.py --quick --model gnb --features both

## Cell 5 — Full run: GNB, feature sets A & B (~10 min, CPU)
Saves: `results/gnb_a_fold00.json` … `gnb_a_fold09.json`  
       `results/gnb_b_fold00.json` … `gnb_b_fold09.json`

In [ ]:
import os
os.environ['FMA_DATA_DIR']   = '/content/data'
os.environ['FMA_RESULT_DIR'] = '/content/results'

!python run_all.py --model gnb --features both

## Cell 6 — Full run: Logistic Regression, feature sets A & B (~20 min, CPU)
Saves: `results/lr_a_fold00.json` … `lr_b_fold09.json`

In [ ]:
import os
os.environ['FMA_DATA_DIR']   = '/content/data'
os.environ['FMA_RESULT_DIR'] = '/content/results'

!python run_all.py --model lr --features both

## Cell 7 — Full run: FT-Transformer, feature sets A & B (~120 min, GPU)
> **Check Cell 3 shows GPU available before running.**  
> Tuned hyperparameter: `d_token` in {64, 128, 256}. dropout fixed at 0.1.  
> Total training runs: 3 configs x inner K=3 x outer K=10 x 2 feature sets = 180 runs.

Saves: `results/ft_a_fold00.json` … `ft_b_fold09.json`

In [ ]:
import os
os.environ['FMA_DATA_DIR']   = '/content/data'
os.environ['FMA_RESULT_DIR'] = '/content/results'

!python run_all.py --model ft --features both

## Cell 8 — Summarise results
Checks three things:
1. **A vs B comparison** — did constructed features improve minority-group recall?
2. **Middle-value requirement** — assignment requires the middle hyperparameter wins most often.
3. **Error bar stability** — std across 10 folds should be reasonable.

In [ ]:
import json, glob, numpy as np
from collections import Counter

MODELS   = ['gnb', 'lr', 'ft']
FEAT_SETS = ['a', 'b']

# Middle hyperparameter value expected to win most often
MIDDLE = {
    'gnb': ('var_smoothing', 1e-4),
    'lr':  ('C', 0.1),
    'ft':  ('d_token', 128),
}

results_table = {}   # (model, feat_set) -> {f1_mean, f1_std, acc_mean, acc_std}

for model in MODELS:
    for feat in FEAT_SETS:
        key = f'{model}_{feat}'
        fold_files = sorted(glob.glob(f'/content/results/{key}_fold*.json'))
        if not fold_files:
            print(f'{key.upper()}: no results yet')
            continue

        records = [json.load(open(fp)) for fp in fold_files]
        f1s  = np.array([r['macro_f1'] for r in records])
        accs = np.array([r['accuracy'] for r in records])

        param_key, mid_val = MIDDLE[model]
        middle_wins = sum(
            1 for r in records
            if abs(r['best_params'].get(param_key, -1) - mid_val) < 1e-9
        )
        pct     = middle_wins / len(records) * 100
        verdict = 'PASS' if pct >= 50 else 'FAIL -- extend hyperparameter range'
        freq    = Counter(str(r['best_params']) for r in records)

        results_table[(model, feat)] = {
            'f1_mean': f1s.mean(), 'f1_std': f1s.std(),
            'acc_mean': accs.mean(), 'acc_std': accs.std(),
            'n_folds': len(records),
        }

        print('=' * 58)
        print(f'  {key.upper()}  ({len(records)}/10 folds complete)')
        print(f'  macro-F1 : {f1s.mean():.4f} +/- {f1s.std():.4f}')
        print(f'  accuracy : {accs.mean():.4f} +/- {accs.std():.4f}')
        print(f'  Middle value ({mid_val}) wins: {middle_wins}/{len(records)} ({pct:.0f}%) -> {verdict}')
        print(f'  Param freq : {dict(freq)}')
        print()

# --- A vs B comparison table ---
print('=' * 58)
print('  A vs B: macro-F1 comparison (feature engineering effect)')
print('=' * 58)
print(f'  {"Model":<6}  {"Feat":<6}  {"macro-F1":>14}  {"Folds":>6}')
print('  ' + '-' * 40)
for model in MODELS:
    for feat in FEAT_SETS:
        r = results_table.get((model, feat))
        if r:
            label = 'original 518' if feat == 'a' else '+121 const.'
            print(f'  {model.upper():<6}  {label:<12}  '
                  f'{r["f1_mean"]:.4f} +/- {r["f1_std"]:.4f}  '
                  f'{r["n_folds"]:>3}/10')
print('=' * 58)

## Cell 9 — Download results

In [ ]:
import shutil
from google.colab import files
import glob, os

# Show what we have before downloading
result_files = sorted(glob.glob('/content/results/*.json'))
print(f'Result files found: {len(result_files)}')
print(f'Expected: 60 files (3 models x 2 feature sets x 10 folds)')
for f in result_files:
    print(f'  {os.path.basename(f)}')

shutil.make_archive('results', 'zip', '/content/results')
files.download('results.zip')
print('\nresults.zip downloaded.')
print('Unzip and place all JSON files into your local results/ folder.')